In [1]:
%load_ext autoreload
%autoreload 2

import os
import time
import tqdm
import numpy as np
import pandas as pd

from sklearn.neighbors import LocalOutlierFactor
import matplotlib.pyplot as plt
import socceraction.spadl as spadl

In [2]:
datafolder = "../data"
fifa2018h5 = os.path.join(datafolder, "spadl-fifa2018.h5")
games = pd.read_hdf(fifa2018h5, key="games")
with pd.HDFStore(fifa2018h5) as store:
    actions = []  #list of DataFrames
    for game in tqdm.tqdm(games.itertuples()):
        game_action = store[f"actions/game_{game.game_id}"]
        game_action = spadl.play_left_to_right(game_action, game.home_team_id)
        game_action["is_home"] = game_action["team_id"] == game.home_team_id
        actions.append(game_action)
    actions = pd.concat(actions)
    actions.drop("original_event_id", axis=1, inplace=True)
    actions = pd.merge(actions, spadl.config.actiontypes_df(), how="left")

64it [00:00, 208.59it/s]


In [3]:
def consolidate(actions):
    #actions.fillna(0, inplace=True)

    #Consolidate corner_short and corner_crossed
    corner_idx = actions.type_name.str.contains("corner")
    actions["type_name"] = actions["type_name"].mask(corner_idx, "corner")

    #Consolidate freekick_short, freekick_crossed, and shot_freekick
    freekick_idx = actions.type_name.str.contains("freekick")
    actions["type_name"] = actions["type_name"].mask(freekick_idx, "freekick")

    #Consolidate keeper_claim, keeper_punch, keeper_save, keeper_pick_up
    keeper_idx = actions.type_name.str.contains("keeper")
    actions["type_name"] = actions["type_name"].mask(keeper_idx, "keeper_action")

    actions["start_x"] = actions["start_x"].mask(actions.type_name == "shot_penalty", 94.5)
    actions["start_y"] = actions["start_y"].mask(actions.type_name == "shot_penalty", 34)

    return actions


actions = consolidate(actions)

def add_noise(actions):
    # Start locations
    start_list = ["cross", "shot", "dribble", "pass", "keeper_action", "clearance", "goalkick"]
    mask = actions["type_name"].isin(start_list)
    noise = np.random.normal(0, 0.5, size=actions.loc[mask, ["start_x", "start_y"]].shape)
    actions.loc[mask, ["start_x", "start_y"]] += noise

    # End locations
    end_list = ["cross", "shot", "dribble", "pass", "keeper_action", "throw_in", "corner", "freekick", "shot_penalty"]
    mask = actions["type_name"].isin(end_list)
    noise = np.random.normal(0, 0.5, size=actions.loc[mask, ["end_x", "end_y"]].shape)
    actions.loc[mask, ["end_x", "end_y"]] += noise

    return actions

actions = add_noise(actions)

In [4]:
actions["angle"] = np.arctan2(actions.end_y - actions.start_y, actions.end_x - actions.start_x)
actions["cos_angle"] = np.cos(actions["angle"])
actions["sin_angle"] = np.sin(actions["angle"])

ACTION_MAP = {'clearance': 0,
              'corner': 1,
              'cross': 2,
              'dribble': 3,
              'freekick': 4,
              'goalkick': 5,
              'keeper_action': 6,
              'pass': 7,
              'shot': 8,
              'throw_in': 9}

actions["action_type_id"] = actions["type_name"].map(ACTION_MAP)

In [5]:
mask = ~actions["type_name"].isin(ACTION_MAP.keys())
actions[mask]["type_name"].unique()

array(['bad_touch', 'foul', 'take_on', 'tackle', 'interception',
       'shot_penalty'], dtype=object)

### Rules for splitting possesion sequences

A new posession sequence is defined when one of the following happens:
- First action of a game period.
- Change of team posession.
- Current actions is a freekick.
- Current action is a goalkick.
- Previous action is a bad touch (not sure about this one)
- Previous action was a shot.
- Previous action was a faul.

In [6]:
actions.columns

Index(['game_id', 'period_id', 'time_seconds', 'team_id', 'player_id',
       'start_x', 'start_y', 'end_x', 'end_y', 'type_id', 'result_id',
       'bodypart_id', 'action_id', 'is_home', 'type_name', 'angle',
       'cos_angle', 'sin_angle', 'action_type_id'],
      dtype='object')

In [15]:
actions["new_poss"] = (
    (actions.index == 0) |
    (actions["game_id"] != actions["game_id"].shift(1)) | #change game
    (actions["period_id"] != actions["period_id"].shift(1)) | #change game period
    (actions["team_id"] != actions["team_id"].shift(1)) | #change team possession
    (actions["type_name"] == "goalkick") |
    (actions["type_name"] == "freekick") |
    (actions["type_name"].shift(1) == "shot") |
    (actions["type_name"].shift(1) == "bad_touch") |
    (actions["type_name"].shift(1) == "foul")
)

In [24]:
actions["poss_id"] = actions["new_poss"].cumsum()
actions["idx_in_seq"] = actions.groupby("poss_id").cumcount()
actions["seq_length"] = (
    actions.groupby("poss_id")["idx_in_seq"]
    .transform("max") + 1
)

In [28]:
# Remove short sequences and non-informative actions

mask_short_seq = (actions["seq_length"] > 2) & (actions["type_name"].isin(ACTION_MAP.keys()))
actions[mask_short_seq]

,game_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,...,cos_angle,sin_angle,action_type_id,new_possession,possession_id,actions_so_far,new_poss,poss_id,idx_in_seq,seq_length
0,7585,1,0.240,769,3445.0,52.0625,34.425,43.3125,33.5750,0,...,-0.995315,-0.096688,7.0,True,1,0,True,1,0,7
1,7585,1,0.480,769,5692.0,43.3125,33.575,44.1875,34.4250,21,...,0.717279,0.696786,3.0,False,1,1,False,1,1,7
2,7585,1,2.120,769,5692.0,44.1875,34.425,40.6875,22.5250,0,...,-0.282166,-0.959366,7.0,False,1,2,False,1,2,7
3,7585,1,3.440,769,5685.0,40.6875,22.525,42.4375,21.6750,21,...,0.899508,-0.436904,3.0,False,1,3,False,1,3,7
4,7585,1,4.200,769,5685.0,42.4375,21.675,56.4375,1.2750,0,...,0.565842,-0.824513,7.0,False,1,4,False,1,4,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128476,7559,2,2941.413,799,5187.0,101.0625,18.275,101.9375,18.2750,21,...,1.000000,0.000000,3.0,False,22416,23,False,22416,23,25
128479,7559,2,2943.213,799,5191.0,97.5625,10.625,94.9375,24.2250,0,...,-0.189517,0.981877,7.0,True,22418,0,True,22418,0,4
128480,7559,2,2944.012,799,5173.0,94.9375,24.225,93.1875,23.3750,21,...,-0.899508,-0.436904,3.0,False,22418,1,False,22418,1,4
128481,7559,2,2944.013,799,5173.0,93.1875,23.375,100.1875,23.3750,0,...,1.000000,0.000000,7.0,False,22418,2,False,22418,2,4


In [27]:
actions[actions["seq_length"] <= 2][["time_seconds","team_id","player_id","type_name","result_id","new_poss","poss_id","idx_in_seq"]]

,time_seconds,team_id,player_id,type_name,result_id,new_poss,poss_id,idx_in_seq
7,8.520,768,3336.0,pass,1,True,2,0
8,11.080,768,3094.0,pass,0,False,2,1
59,99.800,769,5686.0,pass,0,True,7,0
98,159.293,769,6196.0,foul,0,True,11,0
102,173.200,769,3494.0,dribble,1,True,13,0
...,...,...,...,...,...,...,...,...
128334,2700.440,774,5250.0,throw_in,0,True,22404,0
128360,2740.040,774,5250.0,pass,0,True,22407,0
128404,2827.533,799,5173.0,interception,0,True,22412,0
128478,2941.813,774,4063.0,tackle,0,True,22417,0


In [17]:
actions[actions["game_id"]==7585][["time_seconds","team_id","player_id","type_name","result_id","new_poss","poss_id","idx_in_seq"]]

,time_seconds,team_id,player_id,type_name,result_id,new_poss,poss_id,idx_in_seq
0,0.24,769,3445.0,pass,1,True,1,0
1,0.48,769,5692.0,dribble,1,False,1,1
2,2.12,769,5692.0,pass,1,False,1,2
3,3.44,769,5685.0,dribble,1,False,1,3
4,4.20,769,5685.0,pass,1,False,1,4
...,...,...,...,...,...,...,...,...
2286,238.64,768,3532.0,shot_penalty,0,True,424,0
2287,290.20,769,6193.0,shot_penalty,0,True,425,0
2288,337.68,768,3308.0,shot_penalty,1,True,426,0
2289,385.20,769,5688.0,shot_penalty,0,True,427,0


In [ ]:
def _add_sequence_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Given a dataframe with a boolean column `new_poss`,
    add:
        - poss_id
        - idx_in_seq
        - seq_length
    """
    df = df.copy()

    df["poss_id"] = df["new_poss"].cumsum()
    df["idx_in_seq"] = df.groupby("poss_id").cumcount()
    df["seq_length"] = df.groupby("poss_id")["poss_id"].transform("size")

    return df

In [ ]:
def prepare_data(actions: pd.DataFrame, action_map: dict):
    """
    Prepare soccer action data for modeling.

    Steps
    -----
    1. Consolidate and perturb coordinates.
    2. Compute action angle and its cosine/sine.
    3. Map action names to integer ids.
    4. Detect possession boundaries.
    5. Build possession sequence features.
    6. Keep only modeled actions and sequences of length >= 3.
    7. Recompute sequence indexing after filtering.
    8. Return feature matrix X and sequence lengths.

    Returns
    -------
    X : np.ndarray
        Array with columns:
        [action_type_id, start_x, start_y, cos_angle, sin_angle]

    lengths : np.ndarray
        Length of each possession sequence, in possession-id order.
    """
    actions = actions.copy()

    # Basic preprocessing
    actions = consolidate(actions)
    actions = add_noise(actions)

    # Directional features
    dx = actions["end_x"] - actions["start_x"]
    dy = actions["end_y"] - actions["start_y"]
    actions["angle"] = np.arctan2(dy, dx)
    actions["cos_angle"] = np.cos(actions["angle"])
    actions["sin_angle"] = np.sin(actions["angle"])

    # Action id
    actions["action_type_id"] = actions["type_name"].map(action_map)

    # Possession starts
    actions["new_poss"] = (
        (actions.index == 0)
        | (actions["game_id"] != actions["game_id"].shift(1))
        | (actions["period_id"] != actions["period_id"].shift(1))
        | (actions["team_id"] != actions["team_id"].shift(1))
        | (actions["type_name"] == "goalkick")
        | (actions["type_name"] == "freekick")
        | (actions["type_name"].shift(1) == "shot")
        | (actions["type_name"].shift(1) == "bad_touch")
        | (actions["type_name"].shift(1) == "foul")
    )

    # Add initial sequence info
    actions = _add_sequence_columns(actions)

    # Keep only actions modeled in action_map and sequences with length >= 3
    valid_action = actions["type_name"].isin(action_map)
    long_enough_seq = actions["seq_length"] >= 3
    df = actions.loc[valid_action & long_enough_seq].copy()

    # IMPORTANT:
    # after filtering, rebuild new_poss so sequence boundaries are still valid
    # in the filtered dataframe
    df["new_poss"] = (
        (df.index == df.index[0])
        | (df["game_id"] != df["game_id"].shift(1))
        | (df["period_id"] != df["period_id"].shift(1))
        | (df["team_id"] != df["team_id"].shift(1))
        | (df["type_name"] == "goalkick")
        | (df["type_name"] == "freekick")
        | (df["type_name"].shift(1) == "shot")
        | (df["type_name"].shift(1) == "bad_touch")
        | (df["type_name"].shift(1) == "foul")
    )

    df = _add_sequence_columns(df)

    # Feature matrix
    X = df[[
        "action_type_id",
        "start_x",
        "start_y",
        "cos_angle",
        "sin_angle",
    ]].to_numpy()

    # One length per possession
    lengths = (
        df.groupby("poss_id")
        .size()
        .to_numpy()
    )

    return X, lengths
